# EP3 — Segmentación de Pacientes y Predicción de Hospitalización

Este notebook desarrolla una solución de Machine Learning aplicada a un problema de salud: **segmentar pacientes en perfiles clínicos y predecir si un paciente será hospitalizado** a partir de sus indicadores de salud.


> **Nota sobre los datos (importante):** el dataset `pacientes.csv` es una **simulación académica** generada para este trabajo. Los valores se construyeron a partir de cuatro perfiles clínicos coherentes (paciente saludable joven, riesgo metabólico, adulto controlado y mayor frágil), con **ruido controlado** para que no quede artificialmente perfecto. Además, se introdujeron de forma deliberada **valores nulos, filas duplicadas, valores atípicos (outliers) y texto inconsistente**, de modo que la fase de limpieza reproduzca el trabajo real con datos sucios. Se declara explícitamente como datos simulados, siguiendo el mismo criterio de simulación usado en la evaluación anterior. La variable objetivo es `hospitalizado` (0 = no, 1 = sí), por lo que el problema supervisado es de **clasificación**.

Integrantes:

- Nicolas Salas
- Francisco Gaete
- Nicolás Cisternas

## 1. Importar librerías

In [3]:
import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

RANDOM_STATE = 42
print("Librerías importadas correctamente.")

Librerías importadas correctamente.


## 2. Cargar el dataset

Cargamos `pacientes.csv` (ajusta la ruta a la carpeta `/data` del repositorio).

In [4]:
df = pd.read_csv("data\pacientes.csv")
df.head()

<>:1: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<>:1: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
C:\Users\pparr\AppData\Local\Temp\ipykernel_30032\720011257.py:1: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
  df = pd.read_csv("data\pacientes.csv")


,id_paciente,edad,imc,presion_sistolica,presion_diastolica,glucosa,colesterol,trigliceridos,hdl,horas_actividad_semana,cigarrillos_dia,sexo,fuma,antecedentes_familiares,hospitalizado
0,PAC_00001,21.0,18.5,99.9,60.6,72.2,148.8,69.2,79.2,7.9,1.0,M,No,Si,0
1,PAC_00002,50.0,39.1,154.8,103.0,175.2,261.5,305.7,22.2,1.0,19.0,F,Si,No,1
2,PAC_00003,49.0,38.0,161.5,99.4,148.0,254.8,290.2,31.4,2.0,17.0,M,Si,No,1
3,PAC_00004,57.0,32.7,155.5,98.9,NaN,246.9,278.5,31.0,0.4,26.0,F,Si,Si,1
4,PAC_00005,21.0,21.0,110.8,70.3,84.6,NaN,93.6,70.4,7.6,0.0,F,No,No,0


## 3. Inspección inicial del dataset

In [5]:
print("Dimensión del dataset (filas, columnas):")
print(df.shape)

Dimensión del dataset (filas, columnas):
(53300, 15)


In [6]:
print("Información general (tipos y no nulos):")
df.info()

Información general (tipos y no nulos):
<class 'pandas.DataFrame'>
RangeIndex: 53300 entries, 0 to 53299
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id_paciente              53300 non-null  str    
 1   edad                     53300 non-null  float64
 2   imc                      52229 non-null  float64
 3   presion_sistolica        53300 non-null  float64
 4   presion_diastolica       52236 non-null  float64
 5   glucosa                  51964 non-null  float64
 6   colesterol               51172 non-null  float64
 7   trigliceridos            50621 non-null  float64
 8   hdl                      51705 non-null  float64
 9   horas_actividad_semana   50104 non-null  float64
 10  cigarrillos_dia          53300 non-null  float64
 11  sexo                     53300 non-null  str    
 12  fuma                     53300 non-null  str    
 13  antecedentes_familiares  51710 non-null  str   

In [7]:
print("Resumen estadístico de columnas numéricas:")
df.describe()

Resumen estadístico de columnas numéricas:


,edad,imc,presion_sistolica,presion_diastolica,glucosa,colesterol,trigliceridos,hdl,horas_actividad_semana,cigarrillos_dia,hospitalizado
count,53300.000000,52229.000000,53300.00000,52236.000000,51964.000000,51172.000000,50621.000000,51705.000000,50104.000000,53300.000000,53300.000000
mean,49.962195,24.867208,132.86055,79.535956,114.783142,208.707313,157.813168,51.789444,3.274264,6.677486,0.427448
std,23.299972,7.822506,25.25031,14.688509,38.576850,50.080860,96.917124,16.555860,2.997741,9.517527,0.494713
min,3.000000,12.700000,85.80000,50.300000,62.700000,117.500000,5.200000,12.300000,0.000000,0.000000,0.000000
25%,31.000000,18.800000,112.50000,68.400000,87.800000,168.300000,87.600000,36.600000,0.500000,0.000000,0.000000
50%,49.000000,22.700000,127.70000,75.700000,101.500000,200.400000,128.200000,53.000000,2.900000,2.000000,0.000000
75%,57.000000,33.000000,152.20000,94.425000,146.100000,245.900000,231.500000,63.100000,6.000000,14.000000,1.000000
max,999.000000,75.400000,409.70000,117.800000,582.900000,701.200000,1236.100000,91.700000,10.600000,34.000000,1.000000


## 4. Distribución de la variable objetivo

Revisamos el balance entre pacientes hospitalizados y no hospitalizados. Mirar la distribución es una buena práctica inicial: detecta si el problema está muy desbalanceado antes de entrenar.

In [8]:
print("Cantidad por clase:")
print(df["hospitalizado"].value_counts())
print()
print("Proporción:")
print(df["hospitalizado"].value_counts(normalize=True).round(3))

Cantidad por clase:
hospitalizado
0    30517
1    22783
Name: count, dtype: int64

Proporción:
hospitalizado
0    0.573
1    0.427
Name: proportion, dtype: float64


## 5. Detección de duplicados

Revisamos si hay filas repetidas. Comparamos ignorando `id_paciente`, porque ese identificador es único por fila aunque el resto de los datos sea idéntico.

In [9]:
cols_sin_id = [c for c in df.columns if c != "id_paciente"]
duplicados = df.duplicated(subset=cols_sin_id).sum()
print("Cantidad de filas duplicadas (ignorando id_paciente):", duplicados)

Cantidad de filas duplicadas (ignorando id_paciente): 798


## 6. Detección de valores nulos

Revisamos cuántos valores faltantes hay por columna. Los datos reales casi siempre traen vacíos por errores de registro.

In [10]:
print("Valores nulos por columna:")
print(df.isnull().sum())
print("\nTotal de nulos:", df.isnull().sum().sum())

Valores nulos por columna:
id_paciente                   0
edad                          0
imc                        1071
presion_sistolica             0
presion_diastolica         1064
glucosa                    1336
colesterol                 2128
trigliceridos              2679
hdl                        1595
horas_actividad_semana     3196
cigarrillos_dia               0
sexo                          0
fuma                          0
antecedentes_familiares    1590
hospitalizado                 0
dtype: int64

Total de nulos: 14659


## 7. Normalización de texto

Las columnas de texto vienen con inconsistencias (mayúsculas, minúsculas, espacios, sinónimos). Por ejemplo, el sexo aparece como `F`, `f`, ` F `, `femenino`. Las unificamos a un formato estándar.

In [11]:
print("Antes de normalizar:")
print("  sexo:", df["sexo"].unique())
print("  fuma:", df["fuma"].unique())

# Limpiar espacios y unificar a minúsculas, luego mapear a un valor estándar
df["sexo"] = (df["sexo"].astype(str).str.strip().str.lower()
              .map({"f": "F", "femenino": "F", "m": "M", "masculino": "M"}))
df["fuma"] = (df["fuma"].astype(str).str.strip().str.lower()
              .map({"si": "Si", "no": "No"}))

print("\nDespués de normalizar:")
print("  sexo:", df["sexo"].unique())
print("  fuma:", df["fuma"].unique())

Antes de normalizar:
  sexo: <StringArray>
['M', 'F', 'f', 'm', ' F ', ' M ', 'masculino', 'femenino']
Length: 8, dtype: str
  fuma: <StringArray>
['No', 'Si', 'SI', 'no', 'si', 'NO', ' Si', ' No']
Length: 8, dtype: str

Después de normalizar:
  sexo: <StringArray>
['M', 'F']
Length: 2, dtype: str
  fuma: <StringArray>
['No', 'Si']
Length: 2, dtype: str


## 8. Corrección de edades imposibles

Algunas edades son claramente errores de registro (valores como 3, 150 o 999 años). Para un estudio de pacientes adultos, tratamos como inválida cualquier edad fuera del rango 18–110 y la marcamos como nula (se rellenará en el paso siguiente).

In [12]:
import numpy as np
antes = ((df["edad"] < 18) | (df["edad"] > 110)).sum()
df.loc[(df["edad"] < 18) | (df["edad"] > 110), "edad"] = np.nan
print("Edades imposibles corregidas (pasadas a nulo):", int(antes))
print("Rango de edad válido ahora:", df["edad"].min(), "-", df["edad"].max())

Edades imposibles corregidas (pasadas a nulo): 56
Rango de edad válido ahora: 20.0 - 88.0


## 9. Eliminación de columnas sin valor analítico

`id_paciente` solo identifica a la persona (como `id_cliente` en clase: no describe nada clínico), por lo que no sirve para agrupar ni predecir.

In [13]:
df = df.drop(columns=["id_paciente"])
print("Columnas tras la eliminación:", df.shape[1])

Columnas tras la eliminación: 14


## 10. Eliminación de duplicados

Quitamos las filas repetidas detectadas antes.

In [14]:
filas_antes = df.shape[0]
df = df.drop_duplicates().reset_index(drop=True)
print("Filas eliminadas por duplicados:", filas_antes - df.shape[0])
print("Filas restantes:", df.shape[0])

Filas eliminadas por duplicados: 1300
Filas restantes: 52000


## 11. Tratamiento de valores nulos

Rellenamos los nulos en lugar de eliminar filas (perderíamos muchos datos). Para las variables **numéricas** usamos la **mediana** (robusta a outliers); para las **categóricas**, el valor más frecuente (la moda).

In [15]:
# Numéricas -> mediana
num_cols = df.select_dtypes(include=[np.number]).columns
for c in num_cols:
    df[c] = df[c].fillna(df[c].median())

# Categóricas -> moda
cat_cols = df.select_dtypes(include="object").columns
for c in cat_cols:
    df[c] = df[c].fillna(df[c].mode()[0])

print("Nulos restantes tras el relleno:", df.isnull().sum().sum())

Nulos restantes tras el relleno: 0


C:\Users\pparr\AppData\Local\Temp\ipykernel_30032\3527919534.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include="object").columns


## 12. Tratamiento de outliers

Variables clínicas como presión, glucosa o triglicéridos traen valores extremos (errores de medición). Filtramos por el percentil 99 de cada una para que no distorsionen el clustering ni los modelos.

In [16]:
filas_antes = df.shape[0]
for col in ["presion_sistolica", "glucosa", "trigliceridos", "colesterol", "imc"]:
    limite = df[col].quantile(0.99)
    df = df[df[col] <= limite]
df = df.reset_index(drop=True)
print("Filas eliminadas por outliers:", filas_antes - df.shape[0])
print("Filas restantes:", df.shape[0])

Filas eliminadas por outliers: 2480
Filas restantes: 49520


## 13. Verificación posterior a la limpieza

In [17]:
print("Nulos totales:", df.isnull().sum().sum())
print("Duplicados:", df.duplicated().sum())
print("Dimensiones finales:", df.shape)

Nulos totales: 0
Duplicados: 0
Dimensiones finales: (49520, 14)


## 14. Creación de variables nuevas (feature engineering)

Creamos variables categóricas legibles, útiles para los gráficos. Se crean sobre `df` (versión con texto), pero NO se usan como entrada de los modelos.

**1. Rango etario** (Joven / Adulto / Mayor) a partir de `edad`.

In [18]:
def clasificar_edad(x):
    if x < 40:
        return "Joven"
    elif x < 65:
        return "Adulto"
    else:
        return "Mayor"

df["rango_etario"] = df["edad"].apply(clasificar_edad)
print(df["rango_etario"].value_counts())

rango_etario
Adulto    24559
Joven     13673
Mayor     11288
Name: count, dtype: int64


**2. Rango de IMC** agrupado en categorías de salud estándar.

In [19]:
def clasificar_imc(x):
    if x < 18.5:
        return "Bajo peso"
    elif x < 25:
        return "Normal"
    elif x < 30:
        return "Sobrepeso"
    else:
        return "Obesidad"

df["rango_imc"] = df["imc"].apply(clasificar_imc)
print(df["rango_imc"].value_counts())

rango_imc
Normal       22068
Obesidad     11814
Bajo peso    11153
Sobrepeso     4485
Name: count, dtype: int64


## 15. Codificación de categóricas con get_dummies

Para los modelos todo debe ser numérico. Aplicamos one-hot encoding sobre una **copia** (`df_modelo`), dejando `df` intacto con el texto para los gráficos. No codificamos los rangos creados (son para graficar).

In [20]:
df_modelo = df.copy()

categoricas = df_modelo.select_dtypes(include="object").columns.tolist()
print("Columnas categóricas detectadas:", categoricas)

cols_a_codificar = [c for c in categoricas if c not in ["rango_etario", "rango_imc"]]
df_modelo = pd.get_dummies(df_modelo, columns=cols_a_codificar, drop_first=True)
df_modelo = df_modelo.drop(columns=["rango_etario", "rango_imc"])

print("Dimensiones de df_modelo:", df_modelo.shape)
df_modelo.head()

Columnas categóricas detectadas: ['sexo', 'fuma', 'antecedentes_familiares', 'rango_etario', 'rango_imc']
Dimensiones de df_modelo: (49520, 14)


C:\Users\pparr\AppData\Local\Temp\ipykernel_30032\4246762761.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categoricas = df_modelo.select_dtypes(include="object").columns.tolist()


,edad,imc,presion_sistolica,presion_diastolica,glucosa,colesterol,trigliceridos,hdl,horas_actividad_semana,cigarrillos_dia,hospitalizado,sexo_M,fuma_Si,antecedentes_familiares_Si
0,21.0,18.5,99.9,60.6,72.2,148.8,69.2,79.2,7.9,1.0,0,True,False,True
1,50.0,39.1,154.8,103.0,175.2,261.5,305.7,22.2,1.0,19.0,1,False,True,False
2,49.0,38.0,161.5,99.4,148.0,254.8,290.2,31.4,2.0,17.0,1,True,True,False
3,57.0,32.7,155.5,98.9,101.5,246.9,278.5,31.0,0.4,26.0,1,False,True,True
4,21.0,21.0,110.8,70.3,84.6,200.4,93.6,70.4,7.6,0.0,0,False,False,False


## 16. Selección de variables para clustering

Buscamos grupos de pacientes parecidos **sin usar la etiqueta** `hospitalizado`. Usamos los indicadores clínicos numéricos. No incluimos `hospitalizado`: en clustering no existe `y`, buscamos grupos, no predecimos.

In [21]:
variables_cluster = ["edad", "imc", "presion_sistolica", "presion_diastolica",
                     "glucosa", "colesterol", "trigliceridos", "hdl",
                     "horas_actividad_semana", "cigarrillos_dia"]
X_cluster = df[variables_cluster].copy()
print("Variables usadas para clustering:", len(variables_cluster))
X_cluster.head()

Variables usadas para clustering: 10


,edad,imc,presion_sistolica,presion_diastolica,glucosa,colesterol,trigliceridos,hdl,horas_actividad_semana,cigarrillos_dia
0,21.0,18.5,99.9,60.6,72.2,148.8,69.2,79.2,7.9,1.0
1,50.0,39.1,154.8,103.0,175.2,261.5,305.7,22.2,1.0,19.0
2,49.0,38.0,161.5,99.4,148.0,254.8,290.2,31.4,2.0,17.0
3,57.0,32.7,155.5,98.9,101.5,246.9,278.5,31.0,0.4,26.0
4,21.0,21.0,110.8,70.3,84.6,200.4,93.6,70.4,7.6,0.0


## 17. Escalar variables

K-Means calcula distancias. Si una variable tiene números grandes y otra pequeños, la grande domina solo por su escala. `StandardScaler` deja todas en una cancha comparable. No borra información; solo nivela la cancha.

In [22]:
scaler = StandardScaler()
X_cluster_escalado = scaler.fit_transform(X_cluster)
print("Datos escalados. Forma:", X_cluster_escalado.shape)

Datos escalados. Forma: (49520, 10)


## 18. Evaluación de K-Means con varios valores de K

Para cada K calculamos la **Inertia** (compacidad; baja siempre al subir K) y el **Silhouette** (qué tan bien ubicado está cada punto; de -1 a 1, más alto es mejor).

In [23]:
rango_k = range(2, 8)
inercias, silhouettes = [], []
for k in rango_k:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    etiquetas = km.fit_predict(X_cluster_escalado)
    inercias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_cluster_escalado, etiquetas,
                                        sample_size=5000, random_state=RANDOM_STATE))

tabla_k = pd.DataFrame({
    "K": list(rango_k),
    "Inertia": [round(i, 1) for i in inercias],
    "Silhouette": [round(s, 3) for s in silhouettes]
})
print(tabla_k.to_string(index=False))

 K  Inertia  Silhouette
 2 190913.3       0.595
 3  88018.0       0.570
 4  47733.7       0.601
 5  45294.5       0.475
 6  43336.3       0.478
 7  41415.0       0.372


## 19. Gráficos para justificar K

In [24]:
fig_codo = px.line(tabla_k, x="K", y="Inertia",
    title="Método del codo (Inertia por K)",
    labels={"K": "Cantidad de clusters (K)", "Inertia": "Inertia"}, markers=True)
fig_codo.show()

In [25]:
fig_sil = px.line(tabla_k, x="K", y="Silhouette",
    title="Silhouette score por K",
    labels={"K": "Cantidad de clusters (K)", "Silhouette": "Silhouette score"}, markers=True)
fig_sil.show()

### Interpretación de la elección de K

Como vimos en clase, **K lo decide el analista**, justificándolo con un argumento visual (gráficos) y uno numérico (tabla). En este dataset, **K=4 obtiene el mejor Silhouette**, claramente por encima de los demás valores. Esto indica cuatro grupos de pacientes bien diferenciados.

El método del codo también muestra una mejora fuerte hasta K=4 y luego se aplana, lo que refuerza la elección. Por eso K=4 combina el mejor argumento numérico y visual.

## 20. Aplicación final de K-Means con el K elegido

In [26]:
K_FINAL = int(tabla_k.loc[tabla_k["Silhouette"].idxmax(), "K"])
print("K elegido (mejor Silhouette):", K_FINAL)

kmeans_final = KMeans(n_clusters=K_FINAL, random_state=RANDOM_STATE, n_init=10)
df["cluster"] = kmeans_final.fit_predict(X_cluster_escalado)

print("\nCantidad de pacientes por cluster:")
print(df["cluster"].value_counts().sort_index())

sil_final = silhouette_score(X_cluster_escalado, df["cluster"],
                             sample_size=5000, random_state=RANDOM_STATE)
print("\nSilhouette del modelo final:", round(sil_final, 3))

K elegido (mejor Silhouette): 4

Cantidad de pacientes por cluster:
cluster
0    13654
1    11946
2    11300
3    12620
Name: count, dtype: int64

Silhouette del modelo final: 0.601


## 21. Perfil de los clusters

K-Means entrega solo números. Para entenderlos, miramos el promedio de cada variable por grupo.

In [27]:
perfil_clusters = df.groupby("cluster")[variables_cluster].mean().round(1)
perfil_clusters

,edad,imc,presion_sistolica,presion_diastolica,glucosa,colesterol,trigliceridos,hdl,horas_actividad_semana,cigarrillos_dia
cluster,,,,,,,,,,
0,26.0,20.2,105.9,65.2,82.0,154.7,71.6,71.5,7.2,0.9
1,51.5,36.1,159.8,101.5,163.2,266.5,280.1,29.7,0.7,22.1
2,79.4,17.2,143.1,73.1,110.8,219.3,149.8,51.1,0.7,1.7
3,47.0,24.5,120.5,77.4,95.7,189.3,113.5,54.4,3.9,1.3


## 22. Nombres interpretables para los clusters

Los nombres no salen del algoritmo: los construimos mirando los promedios. Calculamos un **indicador de riesgo clínico** (suma de rangos de IMC, presión, glucosa, colesterol, triglicéridos y cigarrillos, menos actividad y HDL), y asignamos nombres de negocio según el perfil.

In [28]:
perfil_nombres = perfil_clusters.copy()
perfil_nombres["indicador_riesgo"] = (
    perfil_nombres["imc"].rank() + perfil_nombres["presion_sistolica"].rank() +
    perfil_nombres["glucosa"].rank() + perfil_nombres["colesterol"].rank() +
    perfil_nombres["trigliceridos"].rank() + perfil_nombres["cigarrillos_dia"].rank() -
    perfil_nombres["horas_actividad_semana"].rank() - perfil_nombres["hdl"].rank()
)

# Ordenamos por riesgo y por edad para nombrar
orden_riesgo = perfil_nombres["indicador_riesgo"].sort_values()
mas_sano = orden_riesgo.index[0]
mas_riesgo = orden_riesgo.index[-1]
# de los dos intermedios, el de mayor edad es "mayor frágil"
intermedios = [c for c in perfil_nombres.index if c not in [mas_sano, mas_riesgo]]
mayor = max(intermedios, key=lambda c: perfil_nombres.loc[c, "edad"])
controlado = [c for c in intermedios if c != mayor][0]

cluster_labels = {
    mas_sano: "Saludable / bajo riesgo",
    mas_riesgo: "Riesgo metabólico alto",
    controlado: "Adulto controlado",
    mayor: "Mayor frágil",
}
df["segmento"] = df["cluster"].map(cluster_labels)

tabla_nombres = pd.DataFrame({
    "cluster": list(cluster_labels.keys()),
    "nombre_de_negocio": list(cluster_labels.values())
}).sort_values("cluster").reset_index(drop=True)
print(tabla_nombres.to_string(index=False))
print()
print(perfil_nombres.round(1).to_string())

 cluster       nombre_de_negocio
       0 Saludable / bajo riesgo
       1  Riesgo metabólico alto
       2            Mayor frágil
       3       Adulto controlado

         edad   imc  presion_sistolica  presion_diastolica  glucosa  colesterol  trigliceridos   hdl  horas_actividad_semana  cigarrillos_dia  indicador_riesgo
cluster                                                                                                                                                        
0        26.0  20.2              105.9                65.2     82.0       154.7           71.6  71.5                     7.2              0.9              -1.0
1        51.5  36.1              159.8               101.5    163.2       266.5          280.1  29.7                     0.7             22.1              21.5
2        79.4  17.2              143.1                73.1    110.8       219.3          149.8  51.1                     0.7              1.7              12.5
3        47.0  24.5              1

## 23. PCA para visualizar los clusters

PCA reduce las variables a 2 componentes solo para dibujar los clusters. **Recordar (errores típicos de clase):** K-Means crea los clusters; PCA solo los visualiza. PC1 y PC2 son columnas nuevas, no variables originales.

In [29]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
componentes = pca.fit_transform(X_cluster_escalado)
df["PC1"], df["PC2"] = componentes[:, 0], componentes[:, 1]

total_var = pca.explained_variance_ratio_.sum()
print("Variación explicada por PC1 y PC2:", pca.explained_variance_ratio_.round(3))
print(f"PC1 + PC2 resumen el {round(total_var*100, 1)}% de la variación.")
if total_var >= 0.70:
    print("PCA confiable: el gráfico 2D representa muy bien los datos.")
elif total_var >= 0.50:
    print("PCA aceptable: aproximación razonable.")
else:
    print("ATENCIÓN: PC1+PC2 capturan poca variación.")

Variación explicada por PC1 y PC2: [0.746 0.164]
PC1 + PC2 resumen el 91.0% de la variación.
PCA confiable: el gráfico 2D representa muy bien los datos.


## 24. Visualización de los clusters con PCA

Con muchos puntos encimados un scatter se ve saturado, así que graficamos una **muestra aleatoria** con marcadores pequeños. Esto no cambia los datos, solo deja verlos mejor.

In [30]:
muestra = df.sample(n=20000, random_state=RANDOM_STATE)
fig_pca = px.scatter(
    muestra, x="PC1", y="PC2",
    color="segmento",
    title=f"Segmentación de pacientes con PCA (K={K_FINAL}) — muestra 20000",
    labels={"segmento": "Segmento"},
    opacity=0.7
)
fig_pca.update_traces(marker=dict(size=6))
fig_pca.show()

### Interpretación de la segmentación

K-Means encontró **4 segmentos** de pacientes claramente diferenciados, que el gráfico con PCA muestra como cuatro grupos bien separados:

- **Saludable / bajo riesgo:** pacientes jóvenes, IMC normal, presión y glucosa bajas, mucha actividad física, casi sin tabaco.
- **Riesgo metabólico alto:** IMC elevado, presión y glucosa altas, colesterol y triglicéridos altos, HDL bajo, tabaquismo importante. Es el grupo de mayor riesgo.
- **Adulto controlado:** valores clínicos dentro de rangos saludables, riesgo intermedio-bajo.
- **Mayor frágil:** pacientes de edad avanzada, bajo IMC, con algunos valores alterados propios de la edad.

**Decisión de negocio (salud):** cada segmento sugiere una estrategia distinta — prevención para los saludables, intervención intensiva para el riesgo metabólico, seguimiento para los controlados y cuidado geriátrico para los mayores. *El segmento NO indica directamente si el paciente será hospitalizado; eso lo predicen los modelos supervisados.*

## 25. Preparación para modelos supervisados

Ahora usamos la etiqueta `hospitalizado`. Predecimos si el paciente será hospitalizado a partir de sus indicadores. Es un problema de **clasificación**.

In [31]:
X = df_modelo.drop(columns=["hospitalizado"])
y = df_modelo["hospitalizado"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
print("Train:", X_train.shape, "| Test:", X_test.shape)

Train: (37140, 13) | Test: (12380, 13)


## 26. Modelo 1 — Regresión Logística

Aunque se llame "regresión", se usa para **clasificación**: calcula una probabilidad y decide una clase.

In [32]:
modelo_log = LogisticRegression(max_iter=1000)
modelo_log.fit(X_train, y_train)
pred_log = modelo_log.predict(X_test)
acc_log = accuracy_score(y_test, pred_log)
print("Accuracy Regresión Logística:", round(acc_log, 3))

Accuracy Regresión Logística: 0.915


c:\Users\pparr\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 27. Modelo 2 — Árbol de Decisión

`max_depth` limita la profundidad para evitar overfitting.

In [33]:
modelo_arbol = DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)
modelo_arbol.fit(X_train, y_train)
pred_arbol = modelo_arbol.predict(X_test)
acc_arbol = accuracy_score(y_test, pred_arbol)
print("Accuracy Árbol de Decisión:", round(acc_arbol, 3))

Accuracy Árbol de Decisión: 0.915


## 28. Matrices de confusión

Filas = realidad, columnas = predicción. La diagonal concentra los aciertos.

In [34]:
cm_log = confusion_matrix(y_test, pred_log)
fig_cm_log = px.imshow(cm_log, text_auto=True, color_continuous_scale="Viridis",
    labels={"x": "Predicción", "y": "Realidad", "color": "Casos"},
    x=["No hosp.", "Hosp."], y=["No hosp.", "Hosp."],
    title="Matriz de confusión - Regresión Logística")
fig_cm_log.show()

In [35]:
cm_arbol = confusion_matrix(y_test, pred_arbol)
fig_cm_arbol = px.imshow(cm_arbol, text_auto=True, color_continuous_scale="Viridis",
    labels={"x": "Predicción", "y": "Realidad", "color": "Casos"},
    x=["No hosp.", "Hosp."], y=["No hosp.", "Hosp."],
    title="Matriz de confusión - Árbol de Decisión")
fig_cm_arbol.show()

## 29. Comparación Train vs Test (detección de overfitting)

Comparamos accuracy en entrenamiento y prueba: si `train` es mucho mayor que `test`, el modelo memoriza; si son parecidos, es estable.

In [36]:
filas = []
for nombre, modelo in [("Regresión Logística", modelo_log), ("Árbol de Decisión", modelo_arbol)]:
    acc_tr = accuracy_score(y_train, modelo.predict(X_train))
    acc_te = accuracy_score(y_test, modelo.predict(X_test))
    filas.append({"Modelo": nombre, "Accuracy_train": round(acc_tr,3),
                  "Accuracy_test": round(acc_te,3), "Diferencia": round(acc_tr-acc_te,3)})
df_train_test = pd.DataFrame(filas)
print(df_train_test.to_string(index=False))

fig_tt = px.bar(
    df_train_test.melt(id_vars="Modelo", value_vars=["Accuracy_train","Accuracy_test"],
                       var_name="Conjunto", value_name="Accuracy"),
    x="Modelo", y="Accuracy", color="Conjunto", barmode="group",
    title="Comparación Accuracy: Entrenamiento vs Prueba", range_y=[0,1])
fig_tt.show()

             Modelo  Accuracy_train  Accuracy_test  Diferencia
Regresión Logística           0.921          0.915       0.006
  Árbol de Decisión           0.923          0.915       0.007


## 30. Comparación final de modelos

In [37]:
comparacion = pd.DataFrame({
    "Modelo": ["Regresión Logística", "Árbol de Decisión"],
    "Accuracy": [round(acc_log, 3), round(acc_arbol, 3)]})
fig_comp = px.bar(comparacion, x="Modelo", y="Accuracy",
    title="Comparación de accuracy entre modelos", text="Accuracy", range_y=[0, 1])
fig_comp.show()
comparacion

,Modelo,Accuracy
0,Regresión Logística,0.915
1,Árbol de Decisión,0.915


## 31. Visualizaciones interactivas con Plotly

## 32. Visualización 1 — Barras: tasa de hospitalización por segmento

Responde: ¿qué perfil de paciente se hospitaliza más?

In [38]:
tasa_seg = df.groupby("segmento", observed=True)["hospitalizado"].mean().reset_index()
tasa_seg["hospitalizado"] = (tasa_seg["hospitalizado"] * 100).round(1)
tasa_seg = tasa_seg.sort_values("hospitalizado")  # ordenar de menor a mayor

fig1 = px.bar(
    tasa_seg, x="segmento", y="hospitalizado",
    title="Tasa de hospitalización por segmento de paciente",
    labels={"segmento": "Segmento", "hospitalizado": "% hospitalizados"},
    text="hospitalizado", color="hospitalizado",
    color_continuous_scale="Reds")
fig1.update_layout(coloraxis_showscale=False)
fig1.show()


### Interpretación del gráfico de barras

El gráfico ordena los cuatro segmentos por su tasa de hospitalización y muestra una diferencia enorme entre ellos: el segmento **Saludable / bajo riesgo** casi no se hospitaliza (~0.4%), el **Adulto controlado** muy poco (~4%), mientras que el **Mayor frágil** (~72%) y sobre todo el **Riesgo metabólico alto** (~98%) concentran casi todas las hospitalizaciones.

Esto confirma que la segmentación de K-Means tiene sentido clínico: los grupos no solo son distintos en sus variables, sino que se comportan muy distinto frente al desenlace de salud. La acción de negocio es clara: priorizar recursos y prevención en los dos segmentos de alto riesgo.

## 33. Visualización 2 — Dispersión: glucosa vs IMC (según hospitalización)

Responde: ¿los hospitalizados se concentran en valores altos de glucosa e IMC?

In [39]:
muestra_disp = df.sample(n=20000, random_state=RANDOM_STATE)

fig2 = px.scatter(
    muestra_disp, x="trigliceridos", y="glucosa",
    color="segmento",
    title="Triglicéridos vs Glucosa por segmento de paciente — muestra 20000",
    labels={"trigliceridos": "Triglicéridos", "glucosa": "Glucosa", "segmento": "Segmento"},
    opacity=0.5)
fig2.update_traces(marker=dict(size=5))
fig2.show()

### Interpretación del gráfico de dispersión

El gráfico cruza dos marcadores metabólicos clave (triglicéridos y glucosa) y colorea cada punto según su segmento. Se aprecian zonas bien diferenciadas: el segmento **Saludable** se concentra abajo a la izquierda (triglicéridos y glucosa bajos), mientras que el **Riesgo metabólico alto** ocupa la esquina superior derecha (ambos valores elevados). Los segmentos **Adulto controlado** y **Mayor frágil** quedan en zonas intermedias.

Esto muestra visualmente *por qué* K-Means separó los grupos: existe una estructura real en los marcadores metabólicos. La dispersión confirma que glucosa y triglicéridos altos van de la mano y caracterizan al grupo de mayor riesgo.

## 34. Visualización 3 — Líneas: tasa de hospitalización según nivel de glucosa

Responde: ¿cómo aumenta el riesgo de hospitalización a medida que sube la glucosa? Dividimos la glucosa en 10 tramos para ver la tendencia con detalle.

In [40]:
# Tendencia de hospitalización según el nivel de glucosa.
# Dividimos la glucosa en 10 tramos (deciles) para ver una curva detallada.
df["nivel_glucosa"] = pd.qcut(df["glucosa"], 10, labels=False)
tasa_glu = df.groupby("nivel_glucosa", observed=True).agg(
    glucosa_mediana=("glucosa", "median"),
    pct_hosp=("hospitalizado", "mean")).reset_index()
tasa_glu["pct_hosp"] = (tasa_glu["pct_hosp"] * 100).round(1)
tasa_glu["glucosa_mediana"] = tasa_glu["glucosa_mediana"].round(0)

fig3 = px.line(
    tasa_glu, x="glucosa_mediana", y="pct_hosp",
    title="Tasa de hospitalización según el nivel de glucosa",
    labels={"glucosa_mediana": "Glucosa (mediana de cada tramo)", "pct_hosp": "% hospitalizados"},
    markers=True)
fig3.show()

### Interpretación del gráfico de líneas

La curva muestra una relación creciente muy marcada en forma de "S": mientras la glucosa se mantiene en niveles normales (por debajo de ~100), la hospitalización es casi nula (menos del 5%); pero a partir de ahí se dispara rápidamente, superando el 80% en los tramos de glucosa más alta.

Es el gráfico que mejor ilustra una **tendencia continua**: a diferencia de comparar categorías, aquí se ve el punto exacto donde el riesgo empieza a crecer (alrededor de 100-110 de glucosa). Clínicamente, marca un umbral de alerta útil para priorizar seguimiento.

## 35. Mini dashboard — resumen ejecutivo

In [41]:
total = df.shape[0]
hosp = int((df["hospitalizado"] == 1).sum())
no_hosp = int((df["hospitalizado"] == 0).sum())
pct = round(hosp / total * 100, 1)

print("====== RESUMEN DEL PROYECTO ======")
print(f"Total de pacientes:          {total}")
print(f"Hospitalizados:              {hosp} ({pct}%)")
print(f"No hospitalizados:           {no_hosp} ({round(100-pct,1)}%)")
print(f"Edad promedio:               {round(df['edad'].mean(),1)}")
print(f"Segmentos K-Means:           {K_FINAL}")
print(f"Mejor modelo (accuracy test):{'Regresión Logística' if acc_log >= acc_arbol else 'Árbol de Decisión'}")
print("==================================")

====== RESUMEN DEL PROYECTO ======
Total de pacientes:          49520
Hospitalizados:              20437 (41.3%)
No hospitalizados:           29083 (58.7%)
Edad promedio:               49.7
Segmentos K-Means:           4
Mejor modelo (accuracy test):Árbol de Decisión
